# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-09_prompt_vs_rag.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day2/T1-09_prompt_vs_rag.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day2"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day2" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)

# Day 2 helper lives here; Day 3 notebooks that import rag also add ../day2


# T1-09 Prompt vs RAG

**Time:** 12:45–13:45 &nbsp;|&nbsp; **Evidence:** side-by-side log — invented circular vs cited paragraph vs abstain

**Notebook learning outcomes**

- Show the **same question** answered from weights only vs from a retrieved chunk.
- Name **hallucination** when a circular number appears with no source.
- **Abstain** when the corpus has no paragraph (Q-MISSING).


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers

**Prompt-only** here means: the generator never sees a handbook chunk. It continues from the standing instruction plus the user text. That is Day 1.

**RAG** means: retrieve first, then generate from visible chunks.

The failure mode ministries actually fear is not a blank. It is a **fluent wrong**: “CivicDesk Circular 99 of 2024 appears to allow…” — invented, confident, unusable in Finance or Defence.

![Same question, two paths](images/t2_prompt_vs_rag.jpg)

**Extra words:**

- **Grounded** — every factual clause points at a chunk id.
- **Ungrounded / hallucinated** — the clause cannot be found in the prompt or the retrieved set.
- **Abstain** — the honest third output. Not a shrug: a logged stop.


## Ministry link

Finance / Defence / Health cannot accept an invented circular, OPR note, or guideline number. Labour cannot accept an invented C2. Trade cannot accept “your password has been reset” because the model sounded sure.

This lab is **N3**. The abstain on Q-MISSING is the skill to take home.


## 0. Backend — demo, OLMo, or Phi

Retrieval always runs on this machine. This cell only chooses **what generates**.

| `BACKEND` | Use when |
|---|---|
| **`demo`** | No GPU, overnight, or the server is down. Canned / template replies. **Not** a live-model score. |
| **`olmo`** | Local OLMo is up (T1-01a / `docker compose`, port **8000**). Needs VRAM. |
| **`phi`** | Local Phi is up (port **8001**). Compose loads OLMo only unless you set `CIVICDESK_LOAD_MODELS=olmo,phi`. |

Change the one marked line and re-run **this** cell. Later generate cells in this kernel follow it. If the live call fails, the notebook continues with a labelled local fallback.


In [ ]:
# [CHANGE] where generate() goes. Retrieval still runs here either way.
BACKEND = "demo"  # "demo" | "olmo" | "phi"

import civicdesk_rag as rag

rag.set_backend(BACKEND)


## 1. Same question, two replies


In [ ]:
# [RUN]
import civicdesk_rag as rag

q = {x["id"]: x for x in rag.load_questions()}["Q-MISSING"]
print("USER:", q["user"])
print()
print("--- prompt-only ---")
po = rag.generate_prompt_only(q["user"])
print("evidence:", po["evidence"], "mode:", po["mode"])
print(po["reply"])
print()
print("--- RAG ---")
out = rag.generate_with_rag(q["user"], role="clerk")
print("evidence:", out["evidence"], "mode:", out["mode"], "visible:", out["visible_ids"])
print(out["reply"])
if out["mode"] != "live":
    print("not a model answer, not scored")
rag.save_log("T1-09_missing", {"prompt_only": po, "rag": {k: out[k] for k in out if k != "visible_chunks"}})


### Teaching note

Mark a prompt-only circular number with no source as a fail, even if it reads fluently.

Without a matching **RECORDED MODEL RUN**, leave model-performance comparisons unmeasured. Local template continuation still shows the pipeline.


## Run 1: the frozen set

Twelve items. Live only the ids you list. Everything else comes from a replay file if one matches; otherwise **model result unavailable**. Simulations never enter model accuracy totals.


In [ ]:
# [CHANGE] pick two or three ids to attempt live. Leave the rest to replay or unavailable.
LIVE_IDS = ["Q-ATTACH", "Q-MISSING"]
CONFIG_1 = {"retriever": "keyword", "min_score": 2, "current_only": False, "system": "STRICT_SYSTEM", "k": 3}

import civicdesk_rag as rag

run1 = rag.run_known_answer_set(CONFIG_1, live_ids=LIVE_IDS, replay="T1-09_run1.json")
rag.save_log("T1-09_run1_local", {"config": CONFIG_1, "totals": run1["totals"]})


### Teaching note — `got` is not `met`

The table can show **expected `cite`** and **got `cite`** while **met** is still `no`. Those columns are different tests.

| Column | What it is | What it is not |
|---|---|---|
| **expected** | The frozen key for this id (`cite` / `abstain` / `refuse` / `false-premise`) | A judgement of the live sentence |
| **got** | The **keyword template's** label, from retrieval and the access gate | The live model's outcome |
| **met** | Automatic checks on the **reply that was scored** (live or recorded) | Proof the quote supports a hire, a notice, or a reset |

For a **cite** to be `met`, the reply must contain the required handbook id in square brackets (for example `[HB-ATTACH-04]`), must not invent an id that was not in the retrieved chunks, and must not cite a forbidden id. For an **abstain**, the reply must contain `NOT IN SOURCES`. A quoted span that is not word-for-word in a visible chunk is flagged even if the id looks right.

So retrieval can do its job (template says cite) while generation still fails (OLMo cites a phantom id, skips the required id, or paraphrases the handbook). That is a **generation** failure, not a retrieval miss. Only a person can still decide whether a passing quote actually supports the answer.

The next cell re-runs **only** `LIVE_IDS` and prints those checks. Unavailable rows are skipped. The live wording may differ slightly from the table because it is a new call.

In [ ]:
# [RUN] inspect why met is no
import json
import civicdesk_rag as rag

ka = {i["id"]: i for i in rag.load_known_answer_set()["items"]}
print("got/local_outcome = template label. met = checks on this live reply.")
print("A cite can still fail if a required [ID] is missing, an unknown [ID] appears, or a quote is not verbatim.")
print()

for qid in LIVE_IDS:
    item = ka[qid]
    out = rag.generate_with_rag(
        item["user"],
        role=item.get("role", "clerk"),
        k=CONFIG_1["k"],
        min_score=CONFIG_1["min_score"],
        current_only=CONFIG_1["current_only"],
        item=item,
    )
    ch = out["checks"]
    cited = ch.get("cited_ids") or []
    unknown = ch.get("unknown_citations") or []
    must = item.get("must_cite") or []
    must_not = item.get("must_not_cite") or []
    missing = [i for i in must if i not in cited]
    forbidden = [i for i in must_not if i in cited]
    altered = [q for q in ch.get("quotes") or [] if not q.get("verbatim")]

    why = []
    if missing:
        why.append("missing required citation " + ", ".join(missing))
    if unknown:
        why.append("unknown citation " + ", ".join(unknown) + " (not in visible_ids)")
    if forbidden:
        why.append("cited forbidden id " + ", ".join(forbidden))
    if item["expected_outcome"] == "abstain" and not ch.get("abstained"):
        why.append("reply does not contain NOT IN SOURCES")
    if item["expected_outcome"] == "false-premise" and not ch.get("flagged_missing_doc"):
        why.append("did not say the named document is not in the sources")
    if altered:
        why.append("quoted span is not word-for-word in a visible chunk")
    if ch.get("outcome_met"):
        why = ["automatic checks passed"]
        if item["expected_outcome"] == "cite":
            why.append("a person still judges whether the quote supports the answer")

    print("=" * 60, qid)
    print("expected:", item["expected_outcome"], "local_outcome:", out["local_outcome"], "met:", ch.get("outcome_met"))
    print("why:", "; ".join(why))
    print("must_cite:", must, "must_not_cite:", must_not)
    print("visible_ids:", out["visible_ids"])
    print("cited_ids:", cited)
    print("unknown_citations:", unknown)
    print("abstained:", ch.get("abstained"))
    print("quotes:", json.dumps(ch.get("quotes") or [], indent=2))
    print("reply:")
    print(out["reply"])
    print()


## Run 2: change one thing

Change **exactly one** of `min_score`, `current_only`, or the system text. Prepared replay names: `T1-09_run2_current_only.json`, `T1-09_run2_min_score.json`.


In [ ]:
# [CHANGE] exactly one key vs CONFIG_1.
CONFIG_2 = {"retriever": "keyword", "min_score": 2, "current_only": True, "system": "STRICT_SYSTEM", "k": 3}
REPLAY_2 = "T1-09_run2_current_only.json"

import civicdesk_rag as rag

run2 = rag.run_known_answer_set(CONFIG_2, live_ids=[], replay=REPLAY_2)
changed = rag.compare_runs(run1, run2)
stale = rag.load_replay("T1-09_run1.json")
print("stale run1 replay usable under CONFIG_2:", rag.replay_usable(stale, CONFIG_2) if stale else "no replay file (model result unavailable)")
if stale and rag.replay_usable(stale, CONFIG_1) and not rag.replay_usable(stale, CONFIG_2):
    print("stale replay rejected after the setting change")


## Three failures

Noise-hit / paraphrase, stale-rule, Circular 88 false-premise. Write the failing stage: retrieval, source, or generation.


In [ ]:
# [RUN]
import civicdesk_rag as rag

ka = {i["id"]: i for i in rag.load_known_answer_set()["items"]}
for iid in ("Q-PARAPHRASE", "Q-STALE", "Q-FALSE-PREMISE"):
    item = ka[iid]
    print("=" * 60, iid)
    print("user:", item["user"])
    print("expected:", item["expected_outcome"], "stage_if_wrong:", item["stage_if_wrong"])
    print("why:", item["why"])
    print("hits:", [(h["id"], h["score"]) for h in rag.retrieve(item["user"], k=5, current_only=False)])


Your notes: paraphrase failing stage ____ ; stale-rule ____ ; Circular 88 ____


In [ ]:
# [CHANGE] Mark two cite items by hand. yes / partly / no.
SUPPORTED = {"Q-ATTACH": None, "Q-C2": None}

auto_cite_ok = sum(1 for r in run1["rows"] if r.get("expected") == "cite" and r.get("met"))
hand = sum(1 for v in SUPPORTED.values() if str(v).lower() == "yes")
print("automatic cite-met (live+recorded only, and only if scored):", auto_cite_ok)
print("hand-marked supported yes:", hand, "of", len(SUPPORTED))
print("A cite outcome still needs a person to judge whether the quote supports the answer.")


In [ ]:
# [OPTIONAL] cloud baseline. Fictional items only. Free tiers may keep prompts.
print("Optional. Reuse the T1-03 comparison key if you have a live cloud endpoint. Not run in the default room.")
print("If you run it, print totals next to OLMo and label the provider. Do not paste ministry text.")


### After you run

Use the **counts** (met / not met / unavailable / simulated). Do not declare a production verdict from this classroom set. A mixed batch is not a single live benchmark.

**Write-up:** If the chunk is not in the retrieved set, we will not invent a circular.
